## Vehicle Points Tracking

### 1. Point tracking

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys, os

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from tracker import track_vehicle_motion
from estimate_weight import estimate_vehicle_weight, extract_vertical_response
# Path to video
# Example video: Case-3a
video_path = Path('videos/case-3a.mov')

print("Video exists:", video_path.exists())
print(video_path.resolve())
response_dict = track_vehicle_motion(
    video_path=video_path,
    use_subpixel=True,  # subpixel body-point tracking
)


### 2. Plot the results

In [ ]:
import numpy as np
from plot.plot_results import plot_tracking_results
fs_tracking = 120.0
T = response_dict["body_front"].shape[0]
response_dict["time"] = np.arange(T) / fs_tracking

# Plot tracking signals
plot_tracking_results(
    responses_2d=response_dict, # dictionary of tracked points (x,y)
    fs=fs_tracking,
    Start=0.0,
    Measurement_Time=2,
    flip_y=False,
    normalize=False,
)

### 3. Generate videos of tracking

In [ ]:
from plot.video_generation import generate_tracking_video
output_path = generate_tracking_video(
    video_path=video_path,
    responses_2d=response_dict,
    fps=fs_tracking / 2,
)

### 4. Weight Estimation

In [ ]:
# Extract vertical responses from 2D responses for weight estimation
response_dict_vertical = extract_vertical_response(response_dict)
# Estimate weight
estimation = estimate_vehicle_weight(
    response=response_dict_vertical,
    kf=50,#45.6*1.1,
    kr=50,#41*1.1,
    leng=2.5,
    fs=120,
    p=70,
    detrend=False,
    plot_figure=True,
)